## Recurrent Neural Network (RNN)

The aim of this notebook is to show a full example of how to implement from scratch a RNN, first using only NumPy and then using thorcino (this framework), building the necessary component to make it a resable and composable module.

#### Disclaimer

If you have not an experience with RNNs or NN, read the [README](./README.md) it contains a brief introduction to the topic and some resources to learn more about it.

## The Learning Task

Our RNN, will be very simple, (but general enough to be used in other tasks), the learning task is to predict the next number in a sequence of equidistant ordered numbers.

In [32]:
"""Hyper parameters (README for the complete notation)"""

N = 10 # number of samples
T = 5  # length of sequences (time steps)
D = 1  # number of features of each sequence element 
O = 1  # number of output units
H = 1  # number of hidden units


In [40]:
"""Building Dataset"""

import numpy as np

def create_sequence(len: int, start: float, stop: float) -> np.ndarray:
    return np.linspace(start, stop, len)

def create_random_sequences(n: int, len:int, min_start: int=10, seed: float=777) -> np.ndarray:
    rng = np.random.default_rng(seed)
    sequences = []
    for _ in range(n):
        start = rng.integers(min_start, size=1)[0]
        stop = start * 2
        seq = create_sequence(len+1, start, stop)

        sequences.append(seq)

    return np.stack(sequences)

A = create_random_sequences(N, T)
X, Y = A[:, :A.shape[1]-1], A[:, A.shape[1]-1:]

print('input data:')
print(X)

print('\n\ntarget data:')
print(Y)

input data:
[[ 9.  10.8 12.6 14.4 16.2]
 [ 6.   7.2  8.4  9.6 10.8]
 [ 3.   3.6  4.2  4.8  5.4]
 [ 3.   3.6  4.2  4.8  5.4]
 [ 0.   0.   0.   0.   0. ]
 [ 6.   7.2  8.4  9.6 10.8]
 [ 4.   4.8  5.6  6.4  7.2]
 [ 9.  10.8 12.6 14.4 16.2]
 [ 3.   3.6  4.2  4.8  5.4]
 [ 1.   1.2  1.4  1.6  1.8]]


target data:
[[18.]
 [12.]
 [ 6.]
 [ 6.]
 [ 0.]
 [12.]
 [ 8.]
 [18.]
 [ 6.]
 [ 2.]]


In [ ]:
"""Building the RNN state, hidden state, weights and bias"""

from thorcino.functions import mse


SEED = 777

rng = np.random.default_rng(SEED)

h_activation = np.tanh
o_activation = np.identity
loss = mse

W_xh = np.random.randn(D, H) # Input weight matrix (shared across time steps)
W_ho = np.random.randn(H, O) # Output weight matrix (shared across time steps)
W_hh = np.random.randn(H, H) # hidden-state-to-hidden-state matrix
H_t = np.random.randn(N, H)  # hidden state at the input at time step t
b_h = np.random.randn(1, H)  # hidden bias
b_o = np.random.randn(1, O)  # output bias

`compute_H` function computes the hidden state of the RNN as the following equation:

$$
\mathbf{H}_t = \phi_h \left( \mathbf{X}_t \mathbf{W}_{xh} + \mathbf{H}_{t-1} \mathbf{W}_{hh} + \mathbf{b}_h \right)
$$

`compute_output` function computes the output of the RNN as the following equation:
$$
\mathbf{O}_t = \phi_o \left( \mathbf{H}_t \mathbf{W}_{ho} + \mathbf{b}_o \right)
$$

In [42]:
"""forward utilities"""

def compute_H(X: np.ndarray, W_xh:np.ndarray, W_hh:np.ndarray, H_t: np.ndarray, b_h: np.ndarray) -> np.ndarray:
    L = X@W_xh + H_t@W_hh + b_h
    return h_activation( L )

def compute_output(H_t: np.ndarray, W_ho: np.ndarray, b_o) -> np.ndarray:
    L = H_t@W_ho + b_o
    return o_activation( L )


def forward(X: np.ndarray, W_xh:np.ndarray, W_hh:np.ndarray, H_t: np.ndarray, b_h: np.ndarray, W_ho: np.ndarray, b_o) -> tuple[np.ndarray, np.ndarray]:
    cache, loss = [], 0

    for t in range(T-1):
        Hprev = H
        H = compute_H(X, X[t, :], W_xh, W_hh, Hprev, b_h)
        out = compute_output(H, W_ho, b_o)
        loss += loss(out, X[t+1, :])

        cache.append((Hprev, H, out))

    return loss, cache

In [ ]:
"""backward utilities"""

def grad__W_ho(L: np.ndarray, O_t: np.ndarray)